# Three decode paths: memory and speed

| script | path | needs |
|---|---|---|
| `run_vllm.py` | vLLM server over HTTP | server **UP** |
| `run_generate.py` | `model.generate(output_logits=True)` | server **DOWN** |
| `run_manual.py` | hand-written decode loop | server **DOWN** |

Each writes `results/<path>.json`. They are separate processes because the server
reserves VRAM at startup, so it cannot be up while the in-process paths run.

```
# with a vLLM server up (e.g. dxo-orchestration: docker compose --profile gpu up -d)
python run_vllm.py

# with every vLLM server stopped
python run_generate.py
python run_manual.py
```

**Read the memory column carefully.** `decode_mib` = peak allocator high-water mark
minus the level before decoding (i.e. the weights). That isolates what decoding cost
— but only for the two in-process paths. vLLM's allocator is in another process and
claims its slab at startup, so its number is an nvidia-smi sample, not a comparable
peak. Speed is comparable across all three: same model, same prompt, greedy decoding.

In [ ]:
import json
from pathlib import Path

RESULTS = Path("results")  # sits next to this notebook
ORDER = ["vllm", "generate", "manual"]

data = {p.stem: json.loads(p.read_text()) for p in RESULTS.glob("*.json")}
found = [k for k in ORDER if k in data]
missing = [k for k in ORDER if k not in data]

for key in found:
    meta = data[key]["meta"]
    print(f"{key:<9} {data[key]['label']:<34} model={data[key]['model']}")
    print(f"{'':<9} saved {meta['saved_at']}  decoding={meta.get('decoding')}")
    if meta.get("other_gpu_mib_at_start"):
        print(f"{'':<9} NOTE other processes held {meta['other_gpu_mib_at_start']:.0f} MiB")
if missing:
    print(f"\nnot run yet: {missing}  (the cells below just skip them)")

## Speed

All three generate the same tokens (greedy), so tokens/sec is directly comparable.

In [ ]:
budgets = sorted({r["budget"] for key in found for r in data[key]["runs"]})


def cell(key, budget, field):
    for run in data[key]["runs"]:
        if run["budget"] == budget:
            return run.get(field)
    return None


def table(field, unit, fmt="{:.1f}"):
    head = f"{'budget':>7} " + " ".join(f"{k:>14}" for k in found)
    print(head)
    print("-" * len(head))
    for budget in budgets:
        row = f"{budget:>7} "
        for key in found:
            value = cell(key, budget, field)
            row += f"{fmt.format(value) if value is not None else '-':>14} "
        print(row)
    print(f"({unit})")


table("tok_per_s", "tokens per second, higher is better")
print()
table("latency_s", "wall seconds for the whole request", "{:.2f}")

## Memory

`decode_mib` is what decoding cost on top of the weights. The question is how it
scales with the number of steps.

In [ ]:
table("decode_mib", "MiB above the weights; vLLM's is cross-process, see note below")

if "generate" in found and "manual" in found:
    print("\ngenerate() minus manual loop, and what one logit row costs:")
    vocab = data["manual"]["meta"]["vocab"]
    print(f"    {'budget':>7} {'extra MiB':>11} {'per step':>10} {'fp32 row':>10}")
    for budget in budgets:
        g, m = cell("generate", budget, "decode_mib"), cell("manual", budget, "decode_mib")
        if g is None or m is None:
            continue
        steps = cell("manual", budget, "tokens") or 1
        print(f"    {budget:>7} {g - m:>11.1f} {(g - m) / steps:>10.3f} {vocab * 4 / 2**20:>10.3f}")
    print("\n'per step' landing on 'fp32 row' is the mechanism: generate() keeps one")
    print("full-vocab float32 logit row per step until the loop ends.")

In [ ]:
# Both in-process paths also pay for the KV cache, which grows with sequence length
# no matter how the loop is written. Separating the two terms is what makes the
# comparison honest: only the logits term differs between the paths.
if "manual" in found:
    meta = data["manual"]["meta"]
    n_prompt = meta["prompt_tokens"]
    print(f"{'budget':>7} {'seq len':>8} {'manual decode':>14} {'of which KV*':>13}")
    for budget in budgets:
        m = cell("manual", budget, "decode_mib")
        if m is None:
            continue
        seq = n_prompt + (cell("manual", budget, "tokens") or 0)
        # Qwen3-4B: 36 layers, 8 KV heads, head_dim 128, bf16 -> 2 bytes.
        kv = 36 * 2 * 8 * 128 * seq * 2 / 2**20
        print(f"{budget:>7} {seq:>8} {m:>14.1f} {kv:>13.1f}")
    print("\n*computed from the architecture, not measured. The remainder is transient")
    print(" activations plus allocator slack.")

In [ ]:
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
for key in found:
    runs = [r for r in data[key]["runs"] if r.get("tok_per_s")]
    left.plot([r["budget"] for r in runs], [r["tok_per_s"] for r in runs], "o-", label=key)
    mem = [r for r in data[key]["runs"] if r.get("decode_mib") is not None]
    if key != "vllm":  # cross-process number, not on the same axis
        right.plot([r["budget"] for r in mem], [r["decode_mib"] for r in mem], "o-", label=key)

left.set(xlabel="max_tokens", ylabel="tokens / s", title="Speed")
left.set_ylim(bottom=0)
right.set(xlabel="max_tokens", ylabel="MiB above weights", title="Decode memory (in-process only)")
right.set_ylim(bottom=0)
for axis in (left, right):
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

## Why vLLM's memory number is not on the same axis

vLLM claims `--gpu-memory-utilization` of the card when it starts and works inside
that slab, mostly as KV cache. So its per-request delta is ~0 — which is not vLLM
winning the memory comparison, it is the cost being paid up front and becoming
invisible per request. `run_vllm.py` samples the server process with nvidia-smi and
records the caveat in its own `meta`.

The comparison vLLM genuinely contributes here is speed.

In [ ]:
if "vllm" in found:
    meta = data["vllm"]["meta"]
    print(f"server pid {meta.get('server_pid')}  fingerprint {meta.get('fingerprint')}")
    print(f"caveat: {meta.get('memory_caveat')}")
    for run in data["vllm"]["runs"]:
        print(
            f"  budget={run['budget']:>4} server held {run['baseline_mib']} -> "
            f"{run['peak_mib']} MiB  (delta {run['decode_mib']})"
        )
else:
    print("run_vllm.py has not been run; start the server and run it to fill this in.")

In [ ]:
from __future__ import annotations

import json
import platform
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
PATH = "manual"
LABEL = "manual loop (reduce in-loop)"
MODEL = "Qwen/Qwen3-4B-FP8"
PROMPT = "Count from 1 to 300, separated by commas. Do not stop early."
BUDGETS = (32, 128, 512)
TOP_LOGPROBS = 3
# RESULTS = Path(__file__).parent / "results"

In [ ]:
import os
import subprocess

out = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,used_gpu_memory", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
    check=False,
).stdout
mine, total = os.getpid(), 0.0
for line in out.strip().splitlines():
    if not line.strip():
        continue
    pid_text, mib_text = (part.strip() for part in line.split(","))
    if int(pid_text) != mine:
        total += float(mib_text)

In [ ]:
print(f"other processes hold {total:.0f} MiB on this GPU")
if total > 500:
    print("  WARNING: the card is shared. Peaks will be skewed and a long run may OOM.")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype="auto", device_map="cuda").eval()

eos = model.generation_config.eos_token_id
eos_ids = frozenset(eos if isinstance(eos, (list, tuple)) else [eos])

In [ ]:
model.generation_config

In [ ]:
prompt = tok.apply_chat_template(
    [{"role": "user", "content": PROMPT}],
    add_generation_prompt=True,
    tokenize=False,
    enable_thinking=False,
)
enc = tok(prompt, return_tensors="pt").to(model.device)
n_prompt = int(enc.input_ids.shape[1])

In [ ]:
meta = {
        "prompt_tokens": n_prompt,
        "vocab": int(model.config.vocab_size),
        "dtype": str(model.dtype),
        "decoding": "greedy",
        "other_gpu_mib_at_start": total,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "host": platform.node(),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

In [ ]:
def other_gpu_mib() -> float:
    """GPU memory held by OTHER processes.

    Worth printing before measuring: a Jupyter kernel or a stray server sharing
    the card both skews the peak and can OOM a long run outright. The number goes
    into the result file so a surprising measurement can be explained later.
    """
    import os
    import subprocess

    out = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,used_gpu_memory", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=False,
    ).stdout
    mine, total = os.getpid(), 0.0
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        pid_text, mib_text = (part.strip() for part in line.split(","))
        if int(pid_text) != mine:
            total += float(mib_text)
    return total

In [ ]:
import math

In [ ]:
cur, cache = enc.input_ids, None
token_ids: list[int] = []
steps: list[dict] = []

print("init_cur: ",cur)

with torch.inference_mode():
    out = model(
        input_ids=cur, past_key_values=cache, use_cache=True, logits_to_keep=1
    )
    print("out: ",out)
    print("logit_shape: ",out.logits.shape)
    print("past_key_values_shape: ",len(out.past_key_values), out.past_key_values)
    print("past_key_values: ",out.past_key_values)
    cache = out.past_key_values
    logits = out.logits[0, -1].float()  # only the last position predicts
    print("logits: ",logits.shape)
    print(logits.argmax())
    chosen = int(logits.argmax())  # greedy
    print(tok.convert_ids_to_tokens([chosen]))
    logprobs = torch.log_softmax(logits.float(), dim=-1)
    print(logprobs)
    probs = logprobs.exp()
    values, ids = logprobs.topk(TOP_LOGPROBS)
    step_i = {
        "token": tok.decode([chosen]),
        "logprob": float(logprobs[chosen]),
        "entropy": float(-(probs * logprobs).nansum()),
        "top": [
            (tok.decode([int(t)]), float(v))
            for v, t in zip(values.tolist(), ids.tolist(), strict=True)
        ],
    }
    print("token: ",repr(step_i['token']))
    for v, t in zip(values.tolist(), ids.tolist()):
        print(f"{t:>7} {repr(tok.decode([t])):>12}  logprob={v:7.3f}  p={math.exp(v):.4f}")
    steps.append(step_i)
    del out, logits  # <-- the whole point: one row resident, not `budget` rows

    token_ids.append(chosen)
    #if chosen in eos_ids:
    #    break
    cur = torch.tensor([[chosen]], device=model.device, dtype=cur.dtype)
    print("cur_i: ",cur)

In [ ]:
k = cache.layers[0].keys
v = cache.layers[0].values
print(len(cache), k.shape, v.shape, k.dtype)

In [ ]:
import inspect
print(model)                                    # 위 구조 그대로
print(inspect.getsource(type(model.model.layers[0]).forward))   # residual 코드
h = model.model(**enc, output_hidden_states=True).hidden_states
print(len(h), h[0].shape)                       # 37, (1, 31, 2560)  ← 임베딩 + 36층

In [ ]:
import torch
from transformers.masking_utils import create_causal_mask
from transformers.cache_utils import DynamicCache

inner, lm_head = model.model, model.lm_head   # Qwen3Model, Linear(2560→151936)

with torch.inference_mode():
    ids = enc.input_ids                 # (1, 31)
    S = ids.shape[1]

    # 1) 토큰 ID → 2560차원 벡터. 여기까진 위치 정보 없음
    h = inner.embed_tokens(ids)                                    # (1, 31, 2560)

    # 2) 위치 인덱스와 RoPE 테이블(cos, sin). 어텐션 안에서 Q·K를 회전시킬 때 쓰임
    pos = torch.arange(S, device=h.device).unsqueeze(0)          # (1, 31)
    print("pos: ",pos)
    pos_emb = inner.rotary_emb(h, pos)

    # 3) 인과 마스크 — 각 위치가 자기 이전만 보도록. 빼먹으면 미래를 훔쳐봅니다
    cache = DynamicCache(config=model.config)
    mask = create_causal_mask(config=model.config, inputs_embeds=h,
                              attention_mask=None, past_key_values=cache,
                              position_ids=pos)


    # 4) 36개 블록을 순서대로. h가 들어가서 h가 나오는 게 전부 — 폭은 계속 2560
    traj = [h]
    for layer in inner.layers:
        h = layer(h, attention_mask=mask, position_embeddings=pos_emb,
                  position_ids=pos, past_key_values=cache, use_cache=True)
        traj.append(h)
        print([None if l.keys is None else tuple(l.keys.shape) for l in cache.layers])

    # 5) 마지막 정규화
    h = inner.norm(h)
    logits = lm_head(h[:, -1:, :]).float()                         # (1, 1, 151936)

chosen = int(logits[0, -1].argmax())
print("token:", repr(tok.decode([chosen])), "| id:", chosen)
print("traj:", len(traj), traj[0].shape, "| cache:", len(cache), cache.layers[0].keys.shape)

In [ ]:
create_causal_mask??

In [ ]:
DynamicCache??

In [ ]:
inner.layers[0]

In [ ]:
pos

In [ ]:
ids

In [ ]:
inner.embed_tokens

In [ ]:
model.config

In [ ]:
mask

In [ ]:
enc

In [ ]:
ids

In [ ]:
with torch.inference_mode():
    ref = model(input_ids=enc.input_ids, use_cache=False, logits_to_keep=1)
print("diff:", float((logits - ref.logits.float()).abs().max()))

In [ ]:
import numpy as np
with torch.inference_mode():
    nxt = torch.tensor([[chosen]], device=model.device)
    h = inner.embed_tokens(nxt)                                    # (1, 1, 2560)
    pos = torch.tensor([[cache.get_seq_length()]], device=h.device)  # [[31]]
    pos_emb = inner.rotary_emb(h, pos)
    mask = create_causal_mask(config=model.config, inputs_embeds=h,
                              attention_mask=None, past_key_values=cache,
                              position_ids=pos)
    for layer in inner.layers:
        h = layer(h, attention_mask=mask, position_embeddings=pos_emb,
                  position_ids=pos, past_key_values=cache, use_cache=True)
        print([None if l.keys is None else tuple(l.keys.shape) for l in cache.layers])

    # for i in range(len(cache.layers)):
    #     # [batch, seq_len, num_heads, head_dim] 순서로 변경 후 1차원 변환
    #     keys_time_first = cache.layers[i].keys.permute(0, 2, 1, 3).detach().cpu().float().numpy().flatten()

    #     plt.figure(figsize=(14, 4))
    #     plt.plot(keys_time_first, linewidth=0.6, color="tab:blue")
    #     plt.title("Cache Keys (Time/Sequence First Flattened)")
    #     plt.xlabel("Index")
    #     plt.ylabel("Value")
    #     plt.grid(True, linestyle="--", alpha=0.5)
    #     plt.tight_layout()
    #     plt.show()
    #     logits2 = lm_head(inner.norm(h)).float()
    # for i in range(len(cache.layers)):
    #     # 1. Keys 및 Values 가져와서 [batch, seq_len, num_heads, head_dim] 순서로 1차원 변환
    #     keys_time_first = (
    #         cache.layers[i].keys.permute(0, 2, 1, 3).detach().cpu().float().numpy().flatten()
    #     )
    #     values_time_first = (
    #         cache.layers[i].values.permute(0, 2, 1, 3).detach().cpu().float().numpy().flatten()
    #     )
    
    #     # 2. 2행 1열 서브플롯 생성 (위아래 X축 연동: sharex=True)
    #     fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    
    #     # [위 패널] Keys 플롯
    #     axes[0].plot(keys_time_first, linewidth=0.6, color="tab:blue")
    #     axes[0].set_title(f"Layer {i} - Cache Keys (Time/Sequence First Flattened)")
    #     axes[0].set_ylabel("Key Value")
    #     axes[0].grid(True, linestyle="--", alpha=0.5)

    #     # [아래 패널] Values 플롯
    #     axes[1].plot(values_time_first, linewidth=0.6, color="tab:orange")
    #     axes[1].set_title(f"Layer {i} - Cache Values (Time/Sequence First Flattened)")
    #     axes[1].set_xlabel("Index")
    #     axes[1].set_ylabel("Value Value")
    #     axes[1].grid(True, linestyle="--", alpha=0.5)

    #     plt.tight_layout()
    #     plt.show()

    #     # 기존 연산 유지
    #     logits2 = lm_head(inner.norm(h)).float()
    
    
    for i in range(len(cache.layers)):
        # 원본 PyTorch tensor
        keys_tensor = cache.layers[i].keys
        values_tensor = cache.layers[i].values

        # 실제 KV cache 메모리 (현재 dtype 기준)
        keys_cache_mem = (
            keys_tensor.numel() * keys_tensor.element_size() / 1024**2
        )
        values_cache_mem = (
            values_tensor.numel() * values_tensor.element_size() / 1024**2
        )

        # 1. 시각화/FFT용 NumPy 변환
        keys_1d = (
            keys_tensor
            .permute(0, 2, 1, 3)
            .detach().cpu().float().numpy().flatten()
        )

        values_1d = (
            values_tensor
            .permute(0, 2, 1, 3)
            .detach().cpu().float().numpy().flatten()
        )

        # 2. FFT
        keys_fft_complex = np.fft.rfft(keys_1d)
        values_fft_complex = np.fft.rfft(values_1d)

        # magnitude는 plotting용
        keys_fft = np.abs(keys_fft_complex)
        values_fft = np.abs(values_fft_complex)

        freqs_keys = np.fft.rfftfreq(len(keys_1d))
        freqs_values = np.fft.rfftfreq(len(values_1d))

        # 3. Inverse FFT
        keys_ifft = np.fft.irfft(
            keys_fft_complex,
            n=len(keys_1d)
        )

        values_ifft = np.fft.irfft(
            values_fft_complex,
            n=len(values_1d)
        )

        # FFT / IFFT NumPy 배열 메모리
        keys_fft_mem = keys_fft_complex.nbytes / 1024**2
        values_fft_mem = values_fft_complex.nbytes / 1024**2

        keys_ifft_mem = keys_ifft.nbytes / 1024**2
        values_ifft_mem = values_ifft.nbytes / 1024**2

        # 4. 2행 3열 subplot
        fig, axes = plt.subplots(2, 3, figsize=(22, 6))

        # ================= Keys =================

        axes[0, 0].plot(
            keys_1d,
            linewidth=0.6,
            color="tab:blue"
        )
        axes[0, 0].set_title(
            f"Layer {i} - Keys Original\n"
            f"KV Cache: {keys_cache_mem:.2f} MB "
            f"({keys_tensor.dtype})"
        )
        axes[0, 0].set_ylabel("Value")
        axes[0, 0].grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        axes[0, 1].plot(
            freqs_keys,
            keys_fft,
            linewidth=0.6,
            color="tab:blue"
        )
        axes[0, 1].set_title(
            f"Layer {i} - Keys FFT Spectrum\n"
            f"Memory: {keys_fft_mem:.2f} MB "
            f"({keys_fft_complex.dtype})"
        )
        axes[0, 1].set_ylabel("Magnitude")
        axes[0, 1].grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        axes[0, 2].plot(
            keys_ifft,
            linewidth=0.6,
            color="tab:green"
        )
        axes[0, 2].set_title(
            f"Layer {i} - Keys Inverse FFT\n"
            f"Memory: {keys_ifft_mem:.2f} MB "
            f"({keys_ifft.dtype})"
        )
        axes[0, 2].set_ylabel("Value")
        axes[0, 2].grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        # ================= Values =================

        axes[1, 0].plot(
            values_1d,
            linewidth=0.6,
            color="tab:orange"
        )
        axes[1, 0].set_title(
            f"Layer {i} - Values Original\n"
            f"KV Cache: {values_cache_mem:.2f} MB "
            f"({values_tensor.dtype})"
        )
        axes[1, 0].set_xlabel("Index")
        axes[1, 0].set_ylabel("Value")
        axes[1, 0].grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        axes[1, 1].plot(
            freqs_values,
            values_fft,
            linewidth=0.6,
            color="tab:orange"
        )
        axes[1, 1].set_title(
            f"Layer {i} - Values FFT Spectrum\n"
            f"Memory: {values_fft_mem:.2f} MB "
            f"({values_fft_complex.dtype})"
        )
        axes[1, 1].set_xlabel("Frequency")
        axes[1, 1].set_ylabel("Magnitude")
        axes[1, 1].grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        axes[1, 2].plot(
            values_ifft,
            linewidth=0.6,
            color="tab:green"
        )
        axes[1, 2].set_title(
            f"Layer {i} - Values Inverse FFT\n"
            f"Memory: {values_ifft_mem:.2f} MB "
            f"({values_ifft.dtype})"
        )
        axes[1, 2].set_xlabel("Index")
        axes[1, 2].set_ylabel("Value")
        axes[1, 2].grid(
            True,
            linestyle="--",
            alpha=0.5
        )

        plt.tight_layout()
        plt.show()

        # 기존 연산 유지
        logits2 = lm_head(inner.norm(h)).float()

print(repr(tok.decode([int(logits2[0, -1].argmax())])))
print("cache now:", cache.layers[0].keys.shape)